# Experiment B.01 — setup, Experiment A gate, and preflight

In [ ]:
import hashlib,json,os,subprocess
from pathlib import Path
GPU="4"
R=Path.home()/"async-vla-latency-bench"; PY=Path.home()/"venv-stage1-id/bin/python"; P=Path.home()/"LIBERO-plus"; EA=Path.home()/"experiment_a"; OUT=Path.home()/"experiment_b"; OUT.mkdir(exist_ok=True)
GATE=EA/"analysis/experiment_a_to_b_gate.json"; AVAL=EA/"experiment_a_validation.json"
required=(R,P,P/"libero/libero/assets",PY,EA/"experiment_a_episode_results.csv",GATE,AVAL)
missing=[str(path) for path in required if not path.exists()]
if missing: raise SystemExit(f"STOP: missing Experiment B prerequisites: {missing}")
gate=json.loads(GATE.read_text()); validation=json.loads(AVAL.read_text())
assert gate["gate_version"]=="experiment_a_to_b_v1" and gate["experiment_b_dispatch"] is True
assert int(gate["negative_variants"])>=2 and float(gate["mean_interaction"])<0 and gate["validation_status"]=="pass"
assert validation["status"]=="pass" and hashlib.sha256(AVAL.read_bytes()).hexdigest()==gate["validation_sha256"]
env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
subprocess.run([str(PY),"-m","pytest","-q",str(R/"async_vla_benchmark/tests")],cwd=R,env=env,check=True)
line=subprocess.run(["nvidia-smi","-i",GPU,"--query-gpu=name,memory.total,memory.used,utilization.gpu,driver_version","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); print(line)
name,total,used,util,driver=[value.strip() for value in line.split(',')]; assert "A100" in name
if int(used)>=500 or int(util)>=5: raise SystemExit(f"STOP: physical GPU {GPU} is not idle: {line}")
provenance={"repository_sha":"anonymous-source","libero_plus_sha":subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"gpu":line,"gpu_index":GPU,"experiment_a_gate_sha256":hashlib.sha256(GATE.read_bytes()).hexdigest(),"experiment_a_validation_sha256":hashlib.sha256(AVAL.read_bytes()).hexdigest()}
(OUT/"experiment_b_preflight_environment.json").write_text(json.dumps(provenance,indent=2)+"\n"); print("PASS: Experiment B preflight complete; Experiment A dispatch gate verified")